In [280]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [281]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv')
data.drop(columns=['API_UserName','observations','dayofweek','is_weekend','day','month'], inplace=True)
# Ensure the 'date' column is in datetime format
data['date'] = pd.to_datetime(data['date'])
data.head(40)

,date,indicator,seen
0,2025-01-01,146.71.50.198,1
1,2025-01-01,149.36.49.225,1
2,2025-01-01,162.142.125.242,1
3,2025-01-01,162.142.125.247,1
4,2025-01-01,162.142.125.255,1
5,2025-01-01,185.230.63.171,1
6,2025-01-01,23.26.221.12,1
7,2025-01-01,23.26.221.2,1
8,2025-01-01,23.26.221.4,1
9,2025-01-01,34.160.111.145,1


In [282]:
def days_since_last_seen_all(data):
    # Filter only rows where seen == 1
    seen_data = data[data['seen'] == 1]
    
    # Sort the data by indicator and date in descending order
    seen_data = seen_data.sort_values(by=['indicator', 'date'], ascending=[True, False])
    
    # Keep only the most recent entry for each indicator
    seen_data = seen_data.drop_duplicates(subset=['indicator'], keep='first')
    
    # Calculate the difference in days from today for each entry
    today = pd.to_datetime(datetime.now()).normalize()
    seen_data['days_since_last_seen'] = (today - seen_data['date']).dt.days
    
    # Drop rows with NaN values
    seen_data = seen_data.dropna()
    
    return seen_data[['indicator', 'date', 'seen', 'days_since_last_seen']]

# Compute "days since last seen" for all indicators
days_since_last_seen_all_data = days_since_last_seen_all(data)

# Merge the "days since last seen" back into the main dataset
data = data.merge(
    days_since_last_seen_all_data[['indicator', 'days_since_last_seen']],
    on='indicator',
    how='left'
)

# Fill NaN values with a default (e.g., a large number for unseen indicators)
data['days_since_last_seen'] = data['days_since_last_seen'].interpolate(method='linear')
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data['days_since_last_seen_scaled'] = scaler.fit_transform(data[['days_since_last_seen']])

In [283]:
data.isna().sum()

date                           0
indicator                      0
seen                           0
days_since_last_seen           0
days_since_last_seen_scaled    0
dtype: int64

In [284]:
# Ensure the 'date' column is in datetime format
data['date'] = pd.to_datetime(data['date'])

# Keep the last 7 days in test_data
last_7_days = datetime.now() - timedelta(days=7)
# Normalize today's date (removes the time portion)
today = pd.to_datetime(datetime.now()).normalize()

# Filter test_data: only indicators seen today
test_data = data[(data['date'] == today) & (data['seen'] == 1)]
 
test_data = test_data.reset_index(drop=True)

# Use the remaining data for training
data = data[data['date'] < last_7_days]

# Reset the index of the filtered data
data = data.reset_index(drop=True)
# Display the test_data
test_data

,date,indicator,seen,days_since_last_seen,days_since_last_seen_scaled
0,2025-04-18,146.71.50.198,1,0,0.0
1,2025-04-18,162.142.125.242,1,0,0.0
2,2025-04-18,162.142.125.247,1,0,0.0
3,2025-04-18,162.142.125.255,1,0,0.0
4,2025-04-18,185.230.63.171,1,0,0.0
5,2025-04-18,64.64.112.131,1,0,0.0
6,2025-04-18,64.64.112.146,1,0,0.0
7,2025-04-18,185.253.162.21,1,0,0.0
8,2025-04-18,104.21.48.1,1,0,0.0
9,2025-04-18,156.146.63.165,1,0,0.0


In [285]:
# Define the cutoff date (90 days ago from today)
cutoff_date = datetime.now() - timedelta(days=120)

# Convert the date column to datetime format (if not already)
data['date'] = pd.to_datetime(data['date'])

# Filter the data to include only rows within the last 90 days
filtered_data = data[data['date'] >= cutoff_date]
filtered_data

,date,indicator,seen,days_since_last_seen,days_since_last_seen_scaled
0,2025-01-01,146.71.50.198,1,0,0.000000
1,2025-01-01,149.36.49.225,1,8,0.079208
2,2025-01-01,162.142.125.242,1,0,0.000000
3,2025-01-01,162.142.125.247,1,0,0.000000
4,2025-01-01,162.142.125.255,1,0,0.000000
...,...,...,...,...,...
20195,2025-04-11,47.237.115.193,0,4,0.039604
20196,2025-04-11,107.180.119.251,0,3,0.029703
20197,2025-04-11,190.92.174.36,0,1,0.009901
20198,2025-04-11,192.124.249.112,0,1,0.009901


In [286]:
unique_indicators_count = filtered_data['indicator'].nunique()
print(f"Number of unique indicators: {unique_indicators_count}")

Number of unique indicators: 200


In [287]:
target_indicator = '102.129.153.158'
indicator_data = test_data[test_data['indicator'] == target_indicator]
print(indicator_data)


Empty DataFrame
Columns: [date, indicator, seen, days_since_last_seen, days_since_last_seen_scaled]
Index: []


In [288]:
indicator_data = data[data['indicator'] == '102.129.153.158']
seen_counts = indicator_data['seen'].value_counts()
print(seen_counts)



seen
0    99
1     2
Name: count, dtype: int64


In [289]:
# Compute per-indicator rolling 7-day reappearance probability
rolling_prob_list = []

# Process each indicator individually
for indicator, group in filtered_data.groupby('indicator'):
    group = group.sort_values('date').reset_index(drop=True)
    
    # Create target: was it seen 7 days later?
    group['seen_7_days_later'] = group['seen'].shift(-7)
    
    # Rolling mean to estimate probability of reappearance
    group['prob_7_day_reappearance'] = group['seen_7_days_later'].rolling(window=30, min_periods=1).mean()
    
    # Add 'days_since_last_seen' for each row
    group['days_since_last_seen'] = group['date'].apply(
        lambda x: (pd.to_datetime(datetime.now()).normalize() - x).days
    )
    
    # Compute differencing for 'prob_7_day_reappearance'
    group['diff_prob_7_day_reappearance'] = group['prob_7_day_reappearance'].diff()
    
    # Add lagged features
    group['lag_1_prob'] = group['prob_7_day_reappearance'].shift(1)
    group['lag_7_prob'] = group['prob_7_day_reappearance'].shift(7)
    
    # Drop NaN values created by differencing and lagging
    group = group.dropna(subset=['diff_prob_7_day_reappearance', 'lag_1_prob', 'lag_7_prob'])
    
    # Save with indicator name
    group['indicator'] = indicator
    rolling_prob_list.append(group)

# Combine all into one DataFrame
rolling_probs = pd.concat(rolling_prob_list).reset_index(drop=True)

# Ensure 'days_since_last_seen' exists in the main dataset
if 'days_since_last_seen' not in data.columns:
    data['days_since_last_seen'] = data['date'].apply(
        lambda x: (pd.to_datetime(datetime.now()).normalize() - x).days
    )

# Show sample of results
rolling_probs[['date', 'indicator', 'seen', 'prob_7_day_reappearance', 'diff_prob_7_day_reappearance', 'lag_1_prob', 'lag_7_prob', 'days_since_last_seen']].sort_values(
    by='prob_7_day_reappearance', ascending=False
).head(100)

,date,indicator,seen,prob_7_day_reappearance,diff_prob_7_day_reappearance,lag_1_prob,lag_7_prob,days_since_last_seen
817,2025-03-14,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,35
816,2025-03-13,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,36
815,2025-03-12,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,37
813,2025-03-10,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,39
798,2025-02-23,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,54
...,...,...,...,...,...,...,...,...
4905,2025-01-25,162.142.125.255,1,0.920000,0.003333,0.916667,0.944444,83
4892,2025-01-12,162.142.125.255,0,0.916667,0.007576,0.909091,0.800000,96
844,2025-04-10,104.21.48.1,1,0.916667,-0.003333,0.920000,0.933333,8
4904,2025-01-24,162.142.125.255,1,0.916667,-0.039855,0.956522,0.941176,84


In [290]:
rolling_probs.isna().sum()

date                               0
indicator                          0
seen                               0
days_since_last_seen               0
days_since_last_seen_scaled        0
seen_7_days_later               1400
prob_7_day_reappearance            0
diff_prob_7_day_reappearance       0
lag_1_prob                         0
lag_7_prob                         0
dtype: int64

In [291]:
from prophet import Prophet

# Get unique indicators
indicators = rolling_probs['indicator'].unique()

# Store results
forecast_results = []

for ip in indicators:
    ip_data = rolling_probs[rolling_probs['indicator'] == ip].copy()
    
    # Only keep needed columns, drop rows with NaN
    prophet_df = ip_data[['date', 'diff_prob_7_day_reappearance', 'days_since_last_seen', 'lag_1_prob', 'lag_7_prob']].dropna()
    prophet_df = prophet_df.rename(columns={
        'date': 'ds',
        'diff_prob_7_day_reappearance': 'y'  # Use differenced data as the target
    })

    # Skip if not enough history
    if len(prophet_df) < 30:
        continue

    # Cap and floor for logistic growth
    prophet_df['cap'] = 1.0
    prophet_df['floor'] = -1.0  # Adjust floor for differenced data

    try:
        # Define and fit model
        model = Prophet(growth='logistic')
        model.add_regressor('days_since_last_seen')
        model.add_regressor('lag_1_prob')  # Add lagged feature as regressor
        model.add_regressor('lag_7_prob')  # Add lagged feature as regressor
        model.fit(prophet_df)

        # Create future dataframe and extend regressor values
        future = model.make_future_dataframe(periods=7)
        future['cap'] = 1.0
        future['floor'] = -1.0

        # Fill future 'days_since_last_seen' and lagged features with last known values
        future['days_since_last_seen'] = prophet_df['days_since_last_seen'].iloc[-1]
        future['lag_1_prob'] = prophet_df['lag_1_prob'].iloc[-1]
        future['lag_7_prob'] = prophet_df['lag_7_prob'].iloc[-1]

        # Predict
        forecast = model.predict(future)

        # Reverse differencing for predictions
        last_original_value = ip_data['prob_7_day_reappearance'].iloc[-1]
        forecast['yhat'] = forecast['yhat'] + last_original_value

        # Extract 7th day forecast
        day_7_forecast = forecast[['ds', 'yhat']].iloc[-1:].copy()
        day_7_forecast['indicator'] = ip
        forecast_results.append(day_7_forecast)

    except Exception as e:
        print(f"Skipped {ip} due to error: {e}")


# Combine results into one DataFrame
forecast_df = pd.concat(forecast_results).reset_index(drop=True)
forecast_df['yhat_percent'] = (forecast_df['yhat'] * 100).round(2).astype(str) + '%'
forecast_df

08:12:03 - cmdstanpy - INFO - Chain [1] start processing
08:12:04 - cmdstanpy - INFO - Chain [1] done processing
08:12:04 - cmdstanpy - INFO - Chain [1] start processing
08:12:06 - cmdstanpy - INFO - Chain [1] done processing
08:12:06 - cmdstanpy - INFO - Chain [1] start processing
08:12:08 - cmdstanpy - INFO - Chain [1] done processing
08:12:08 - cmdstanpy - INFO - Chain [1] start processing
08:12:09 - cmdstanpy - INFO - Chain [1] done processing
08:12:10 - cmdstanpy - INFO - Chain [1] start processing
08:12:12 - cmdstanpy - INFO - Chain [1] done processing
08:12:12 - cmdstanpy - INFO - Chain [1] start processing
08:12:13 - cmdstanpy - INFO - Chain [1] done processing
08:12:14 - cmdstanpy - INFO - Chain [1] start processing
08:12:15 - cmdstanpy - INFO - Chain [1] done processing
08:12:16 - cmdstanpy - INFO - Chain [1] start processing
08:12:17 - cmdstanpy - INFO - Chain [1] done processing
08:12:17 - cmdstanpy - INFO - Chain [1] start processing
08:12:19 - cmdstanpy - INFO - Chain [1]

,ds,yhat,indicator,yhat_percent
0,2025-04-18,0.051339,102.129.153.158,5.13%
1,2025-04-18,-0.008313,102.129.153.43,-0.83%
2,2025-04-18,-0.007975,102.129.153.71,-0.8%
3,2025-04-18,0.130701,102.165.16.161,13.07%
4,2025-04-18,-0.003445,104.160.6.2,-0.34%
...,...,...,...,...
195,2025-04-18,0.000858,international.standardbank.com/,0.09%
196,2025-04-18,0.246907,pub.marq.com/,24.69%
197,2025-04-18,0.134554,realinvestmentadvice.com/,13.46%
198,2025-04-18,-0.014159,www.emergencylighting.com/,-1.42%


In [292]:
# Extract the indicators from both dataframes
sorted_results_indicators = set(forecast_df['indicator'])
test_data_indicators = set(test_data['indicator'])

# Find matching indicators
matching_indicators = sorted_results_indicators.intersection(test_data_indicators)

# Find missing indicators in test_data
missing_in_test_data = sorted_results_indicators.difference(test_data_indicators)

# Find missing indicators in sorted_results_df
missing_in_sorted_results = test_data_indicators.difference(sorted_results_indicators)

# Display the results
print("Matching Indicators:", matching_indicators)
print("Indicators in sorted_results_df but missing in test_data:", missing_in_test_data)
print("Indicators in test_data but missing in sorted_results_df:", missing_in_sorted_results)

Matching Indicators: {'162.142.125.247', '162.142.125.255', '68.67.179.164', '104.21.48.1', '64.64.112.131', '146.71.50.198', '185.230.63.171', '185.253.162.21', '64.64.112.146', '156.146.63.165', '162.142.125.242'}
Indicators in sorted_results_df but missing in test_data: {'104.21.54.132', '46.246.8.22', '23.26.221.19', '104.18.32.191', '46.246.8.17', '46.246.8.46', '23.26.221.9', '64.64.112.156', '216.218.130.2', '156.146.63.173', '64.64.112.158', '46.246.8.84', '156.146.63.169', '23.26.221.23', '192.124.249.112', '102.129.153.43', '23.26.221.10', '23.26.221.17', '46.246.8.144', '104.160.6.2', '23.26.221.7', '46.246.8.32', '156.146.63.166', '162.241.225.237', '23.26.221.14', '88.119.174.148', '3.14.182.203', '46.246.8.116', '46.246.8.122', '46.246.8.119', '46.246.8.5', '46.246.8.85', '64.64.112.157', 'hcmiu.edu.vn/', '172.240.108.68', '107.180.119.251', 'pub.marq.com/', '146.70.204.179', '34.160.111.145', 'realinvestmentadvice.com/', '165.231.34.106', '46.246.8.91', '64.64.112.145', 

In [293]:
precision = len(matching_indicators) / len(sorted_results_indicators)
recall = len(matching_indicators) / len(test_data_indicators)
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")


Precision: 0.06
Recall: 1.00
F1 Score: 0.10


In [294]:
# Filter the missing indicators from forecast_df
missing_indicators_data = forecast_df[forecast_df['indicator'].isin(missing_in_test_data)]

# Display the results
missing_indicators_data[['indicator', 'yhat_percent']]

,indicator,yhat_percent
0,102.129.153.158,5.13%
1,102.129.153.43,-0.83%
2,102.129.153.71,-0.8%
3,102.165.16.161,13.07%
4,104.160.6.2,-0.34%
...,...,...
195,international.standardbank.com/,0.09%
196,pub.marq.com/,24.69%
197,realinvestmentadvice.com/,13.46%
198,www.emergencylighting.com/,-1.42%


In [295]:
# Filter the matching indicators from forecast_df
matching_indicators_data = forecast_df[forecast_df['indicator'].isin(matching_indicators)]

# Get the list of indicators from matching_indicators_data
matching_indicators = matching_indicators_data['indicator'].unique()

# Filter the last 90 days of data for these indicators where seen = 1
seen_in_last_90_days = filtered_data[(filtered_data['indicator'].isin(matching_indicators)) & (filtered_data['seen'] == 1)]

# Get the list of indicators that have been seen
seen_indicators = seen_in_last_90_days['indicator'].unique()

# Find indicators that have not been seen in the last 90 days
not_seen_indicators = set(matching_indicators) - set(seen_indicators)

# Display the indicators not seen in the last 90 days
if not_seen_indicators:
    print("The following indicators have NOT been seen in the last 90 days:")
    print(not_seen_indicators)
else:
    print("All matching indicators have been seen in the last 90 days.")

# Exclude not_seen_indicators from the display list
display_indicators = set(matching_indicators) - not_seen_indicators

# Display the results
matching_indicators_data[matching_indicators_data['indicator'].isin(display_indicators)][['indicator', 'yhat_percent']]


All matching indicators have been seen in the last 90 days.


,indicator,yhat_percent
8,104.21.48.1,88.43%
16,146.71.50.198,26.11%
30,156.146.63.165,27.21%
50,162.142.125.242,82.75%
51,162.142.125.247,75.51%
52,162.142.125.255,75.01%
65,185.230.63.171,77.04%
66,185.253.162.21,41.83%
167,64.64.112.131,7.41%
169,64.64.112.146,30.68%


In [296]:
# Filter records with yhat_percent above 70% and exclude not_seen_indicators
filtered_results = forecast_df[
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) > 70) & 
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  # Use .copy() to avoid SettingWithCopyWarning

# Add a column to indicate whether the prediction was "Right" or "Wrong"
# Assuming you have a ground truth column 'actual' to compare with 'yhat'
# Replace 'actual' with the appropriate column name in your dataset
filtered_results.loc[:, 'prediction_status'] = filtered_results.apply(
    lambda row: 'Right' if row['yhat'] >= row.get('actual', 0) else 'Wrong',
    axis=1
)

# Identify false negatives: indicators in matching_indicators_data but below the threshold
false_negatives = forecast_df[
    (forecast_df['indicator'].isin(matching_indicators_data['indicator'])) & 
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) <= 70) &
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  
false_negatives.loc[:, 'prediction_status'] = 'False Negative'

# Identify false positives: indicators not in matching_indicators_data but meet the threshold
false_positives = forecast_df[
    (~forecast_df['indicator'].isin(matching_indicators_data['indicator'])) & 
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) > 70) &
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  # Use .copy() to avoid SettingWithCopyWarning
false_positives.loc[:, 'prediction_status'] = 'False Positive'

# Combine filtered results, false negatives, and false positives
final_results = pd.concat([filtered_results, false_negatives, false_positives])

# Display the final results
final_results[['indicator', 'yhat_percent', 'prediction_status']]

,indicator,yhat_percent,prediction_status
8,104.21.48.1,88.43%,Right
9,104.21.54.132,77.06%,Right
50,162.142.125.242,82.75%,Right
51,162.142.125.247,75.51%,Right
52,162.142.125.255,75.01%,Right
58,172.240.108.68,80.54%,Right
65,185.230.63.171,77.04%,Right
117,34.160.111.145,86.81%,Right
178,68.67.179.164,81.02%,Right
16,146.71.50.198,26.11%,False Negative


In [297]:
# Calculate accuracy of final results
# Accuracy = (Number of correct predictions) / (Total number of predictions)
correct_predictions = final_results[final_results['prediction_status'] == 'Right'].shape[0]
total_predictions = final_results.shape[0]
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

# Display the accuracy
print (f'{correct_predictions} / {total_predictions}')
print(f"Accuracy: {accuracy:.2%}")

9 / 17
Accuracy: 52.94%
